**Extracción de datos a través de CSV**

In [ ]:
import pandas as pd
import json
import ast
import re

def distill_analysis_gdl(reason_str, player, move_gdl):
    """
    Destilación de alta densidad (20-50 tokens).
    """
    # Limpieza inicial para manejar diversos formatos de modelos
    reason_low = str(reason_str).lower() if pd.notna(reason_str) else ""
    player_name = str(player).lower()
    opp = "black" if player_name == "red" else "red"
    col = str(move_gdl[1])

    # 1. VICTORIA PROPIA (Métricas: win, victory, complete, 4 in a row, goal 100)
    if any(w in reason_low for w in ["win", "victory", "complete", "4 in a row", "four", "100"]):
        analysis = (f"I have three connected pieces aligned. Dropping a piece in column {col} "
                    f"completes the line of four to achieve goal({player_name},100) and a terminal state. "
                    f"The move is decisive under GDL win conditions and gravity rules.")

    # 2. BLOQUEO / PREVENCIÓN (Métricas: block, prevent, stop, threat, respond, must)
    elif any(w in reason_low for w in ["block", "prevent", "stop", "threat", "respond", "must", "opp"]):
        analysis = (f"The opponent ({opp}) has three connected pieces. If I do not act, they will "
                    f"complete their line in column {col} next turn. Dropping here blocks the threat "
                    f"while maintaining my defensive structure under gravity rules.")

    # 3. SETUPS Y TRAMPAS (Métricas: setup, trap, diagonal, three, 3 in a row, pressure)
    elif any(w in reason_low for w in ["setup", "trap", "align", "diagonal", "three", "3 in a row", "pressure", "threaten"]):
        analysis = (f"Strategic alignment detected. Selecting column {col} creates a strong three-in-a-row "
                    f"threat. This forces {opp} to respond defensively while I develop multiple "
                    f"potential lines in the upper rows under gravity.")

    # 4. CONTROL DEL EJE CENTRAL (Métricas: center, middle, central, axis o columnas 4/5)
    elif any(w in reason_low for w in ["center", "middle", "central", "axis"]) or col in ['4', '5']:
        analysis = (f"Column {col} belongs to the central axis. Controlling the center maximizes "
                    f"connectivity for horizontal and diagonal lines, which is fundamental for long-term "
                    f"GDL positional strategy and future threats.")

    # 5. DESARROLLO POSICIONAL (Default)
    else:
        analysis = (f"Analyzing GDL board predicates. Column {col} is the optimal choice to maintain positional "
                    f"tension. This move restricts the opponent's vertical development and improves my "
                    f"overall connectivity across the 8x6 grid.")

    return f"Analysis: {analysis}"

def generate_expert_dataset_gdl(input_excel, output_jsonl):
    print(f"Generando Dataset Experto Connect4")

    # Cargar datos (Hoja por defecto o nombre específico)
    df = pd.read_excel(input_excel)

    # Filtro: Solo jugadas válidas
    df_filtered = df[df['valid'] == 1].copy()

    dataset = []
    system_content = ("You are an expert Connect 4 strategist. Analyze the GDL board state and play optimally to win, "
                      "following strategic rules and maximizing your victory probability.")

    for _, row in df_filtered.iterrows():
        try:
            # 1. Extraer datos Post-jugada del Excel
            post_board_gdl = ast.literal_eval(str(row['board']))
            move_gdl = ast.literal_eval(str(row['move'])) # ['drop', 'X']
            player = str(row['player']).lower()
            target_col = str(move_gdl[1])

            # 2. TRANSFORMACIÓN A ESTADO PRE-JUGADA
            # Identificamos la ficha más alta en la columna del movimiento para removerla
            pre_board_gdl = []
            col_cells = [item for item in post_board_gdl if item[0] == 'cell' and str(item[1]) == target_col]

            # La ficha recién puesta es la de mayor altura (Y) en esa columna
            target_to_remove = sorted(col_cells, key=lambda x: int(x[2]), reverse=True)[0] if col_cells else None

            for item in post_board_gdl:
                if item == target_to_remove:
                    continue # Revertimos la ficha (vuelta al pasado)

                # En el estado PRE, el control es de quien va a mover
                if item[0] == 'control':
                    pre_board_gdl.append(['control', player])
                else:
                    pre_board_gdl.append(item)

            # 3. Procesar LegalMoves y Razón (independiente del modelo)
            legal_moves_gdl = ast.literal_eval(str(row['legalMoves']))

            reason_raw = row['reason']
            # Parser robusto para JSON o texto plano de diferentes modelos
            try:
                if isinstance(reason_raw, str) and reason_raw.strip().startswith('{'):
                    reason_dict = json.loads(reason_raw)
                    raw_explanation = reason_dict.get("explanation", reason_dict.get("reason", str(reason_raw)))
                elif isinstance(reason_raw, str) and reason_raw.strip().startswith('['):
                    raw_explanation = str(ast.literal_eval(reason_raw))
                else:
                    raw_explanation = str(reason_raw)
            except:
                raw_explanation = str(reason_raw)

            # 4. Formatear para Alpaca
            user_input = f"GDL Board State: {pre_board_gdl}\nLegal Moves: {legal_moves_gdl}\nAction for Player: {player.upper()}"

            # Análisis optimizado (20-50 tokens)
            analysis_cot = distill_analysis_gdl(raw_explanation, player, move_gdl)
            assistant_output = f"{analysis_cot}\nMove: {move_gdl}"

            dataset.append({
                "instruction": system_content,
                "input": user_input,
                "output": assistant_output
            })

        except Exception as e:
            continue

    # Guardar JSONL
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"¡Dataset generado con éxito! Total: {len(dataset)} ejemplos.")

if __name__ == "__main__":
    # Ruta de tu archivo Excel
    ruta_input = '/content/drive/MyDrive/TESIS/Qwen_Connect4_NotConnect4/Consolidated_Move_Records_en_tabla_Connect4.xlsx'
    generate_expert_dataset_gdl(ruta_input, 'D1_Expert_Connect4.jsonl')

Generando Dataset Experto Connect4
¡Dataset generado con éxito! Total: 2351 ejemplos.


**Extracción de datos a través de MiniMax**

In [ ]:
import json
import random
import copy
import numpy as np

# --- CONFIGURACIÓN GDL (8x6) ---
COLUMNS = 8
ROWS = 6
EMPTY = 0
RED = 1   # Jugador 'red' (X)
BLACK = 2 # Jugador 'black' (O)

class Connect4Minimax:
    def __init__(self):
        self.board = np.zeros((ROWS, COLUMNS), dtype=int)

    def get_legal_moves(self):
        return [c for c in range(COLUMNS) if self.board[ROWS-1][c] == EMPTY]

    def drop_piece(self, col, piece):
        for r in range(ROWS):
            if self.board[r][col] == EMPTY:
                self.board[r][col] = piece
                return r
        return -1

    def check_win(self, piece):
        # Horizontal
        for c in range(COLUMNS-3):
            for r in range(ROWS):
                if all(self.board[r][c+i] == piece for i in range(4)): return True
        # Vertical
        for c in range(COLUMNS):
            for r in range(ROWS-3):
                if all(self.board[r+i][c] == piece for i in range(4)): return True
        # Diagonales
        for c in range(COLUMNS-3):
            for r in range(ROWS-3):
                if all(self.board[r+i][c+i] == piece for i in range(4)): return True
            for r in range(3, ROWS):
                if all(self.board[r-i][c+i] == piece for i in range(4)): return True
        return False

    def evaluate_board(self, piece):
        score = 0
        opp = BLACK if piece == RED else RED

        # 1. Control de zona central ponderada (columnas 2-7 en 0-indexed)
        center_weights = {2: 1, 3: 2, 4: 3, 5: 3, 6: 2, 7: 1}
        for col, w in center_weights.items():
            col_array = [int(self.board[r][col]) for r in range(ROWS)]
            score += col_array.count(piece) * w

        # 2. Función auxiliar: puntuar una ventana de 4 celdas
        def score_window(window):
            s = 0
            pc = window.count(piece)
            ec = window.count(EMPTY)
            oc = window.count(opp)
            if   pc == 4:             s += 100   # Victoria garantizada
            elif pc == 3 and ec == 1: s += 10    # 3 en línea abierta
            elif pc == 2 and ec == 2: s += 2     # 2 en línea abierta
            if   oc == 3 and ec == 1: s -= 15    # Amenaza rival crítica
            elif oc == 2 and ec == 2: s -= 3     # Amenaza rival leve
            return s

        # 3. Ventanas horizontales
        for r in range(ROWS):
            for c in range(COLUMNS - 3):
                window = [int(self.board[r][c+i]) for i in range(4)]
                score += score_window(window)

        # 4. Ventanas verticales
        for c in range(COLUMNS):
            for r in range(ROWS - 3):
                window = [int(self.board[r+i][c]) for i in range(4)]
                score += score_window(window)

        # 5. Ventanas diagonales /
        for c in range(COLUMNS - 3):
            for r in range(ROWS - 3):
                window = [int(self.board[r+i][c+i]) for i in range(4)]
                score += score_window(window)

        # 6. Ventanas diagonales \
        for c in range(COLUMNS - 3):
            for r in range(3, ROWS):
                window = [int(self.board[r-i][c+i]) for i in range(4)]
                score += score_window(window)

        return score

    def minimax(self, depth, alpha, beta, maximizingPlayer, piece):
          valid_moves = self.get_legal_moves()
          opp = BLACK if piece == RED else RED
          is_terminal = self.check_win(RED) or self.check_win(BLACK) or len(valid_moves) == 0

          if depth == 0 or is_terminal:
              if self.check_win(piece): return (None, 100000 + depth) # Prefiere ganar rápido
              if self.check_win(opp):   return (None, -100000 - depth) # Retrasa la derrota
              return (None, self.evaluate_board(piece))

          if maximizingPlayer:
              value = -np.inf
              best_moves = []

              for col in valid_moves:
                  b_copy = copy.deepcopy(self)
                  b_copy.drop_piece(col, piece)
                  new_score = b_copy.minimax(depth-1, alpha, beta, False, piece)[1]

                  if new_score > value:
                      value = new_score
                      best_moves = [col]
                  elif new_score == value:
                      best_moves.append(col)

                  alpha = max(alpha, value)
                  if alpha >= beta: break

              return random.choice(best_moves) if best_moves else random.choice(valid_moves), value

          else:
              value = np.inf
              best_moves = []

              for col in valid_moves:
                  b_copy = copy.deepcopy(self)
                  b_copy.drop_piece(col, opp)
                  new_score = b_copy.minimax(depth-1, alpha, beta, True, piece)[1]

                  if new_score < value:
                      value = new_score
                      best_moves = [col]
                  elif new_score == value:
                      best_moves.append(col)

                  beta = min(beta, value)
                  if alpha >= beta: break

              return random.choice(best_moves) if best_moves else random.choice(valid_moves), value


def get_gdl_board(matrix):
    """Transforma la matriz en [['cell', 'X', 'Y', 'P'], ...]"""
    gdl = []
    for r in range(ROWS):
        for c in range(COLUMNS):
            val = matrix[r][c]
            if val != EMPTY:
                p = "red" if val == RED else "black"
                gdl.append(['cell', str(c+1), str(r+1), p])
    return gdl


def generate_cot(board_engine, move_col, player):
    """Genera un CoT denso (20-50 tokens) basado en reglas GDL."""
    col_gdl = move_col + 1
    opp = "black" if player == "red" else "red"
    piece     = RED   if player == "red" else BLACK
    opp_piece = BLACK if player == "red" else RED

    # 1. Victoria inmediata
    engine_copy = copy.deepcopy(board_engine)
    engine_copy.drop_piece(move_col, piece)
    if engine_copy.check_win(piece):
        return (f"Analysis: Winning opportunity detected. Dropping a piece in column {col_gdl} "
                f"completes a line of four, reaching goal({player},100) and a terminal state "
                f"under gravity and GDL rules.")

    # 2. Bloqueo crítico
    engine_opp = copy.deepcopy(board_engine)
    engine_opp.drop_piece(move_col, opp_piece)
    if engine_opp.check_win(opp_piece):
        return (f"Analysis: The opponent ({opp}) is threatening to complete a line. Selecting "
                f"column {col_gdl} is mandatory to block their alignment and prevent a goal(0) "
                f"outcome in the next state.")

    # 3. Zona central (columnas 3-6 en 1-indexed)
    if col_gdl in [3, 4, 5, 6]:
        return (f"Analysis: Column {col_gdl} belongs to the central zone. Occupying this "
                f"position maximizes connectivity for horizontal and diagonal lines, "
                f"improving long-term GDL positional strategy.")

    # 4. Desarrollo táctico (flancos)
    return (f"Analysis: Analyzing GDL board predicates. Column {col_gdl} is the optimal choice "
            f"to maintain board tension and improve connectivity across the 8x6 grid "
            f"following gravity mechanics.")


def generate_dataset(num_games=250, depth=4):
    dataset = []
    system_instr = ("You are an expert Connect 4 strategist. Analyze the GDL board state and "
                    "play optimally to win, following strategic rules and maximizing your "
                    "victory probability.")

    for game_idx in range(num_games):
        game = Connect4Minimax()
        turn = RED

        while not (game.check_win(RED) or game.check_win(BLACK) or
                   len(game.get_legal_moves()) == 0):

            current_role = "red" if turn == RED else "black"
            valid_moves  = game.get_legal_moves()

            # Si el tablero está vacío, obligamos a jugar en una de las 4 columnas centrales al azar
            if len(valid_moves) == COLUMNS and all(game.board[0][c] == EMPTY for c in range(COLUMNS)):
                best_col = random.choice([2, 3, 4, 5])
            else:
                best_col, _ = game.minimax(
                    depth, -np.inf, np.inf, True,
                    RED if turn == RED else BLACK
                )
                if best_col is None: break

            gdl_legals        = [['drop', str(m+1)] for m in sorted(valid_moves)]
            current_gdl_board = get_gdl_board(game.board)
            current_gdl_board.append(['control', current_role])

            cot      = generate_cot(game, best_col, current_role)
            move_gdl = ['drop', str(best_col + 1)]

            dataset.append({
                "instruction": system_instr,
                "input":  (f"GDL Board State: {current_gdl_board}\n"
                           f"Legal Moves: {gdl_legals}\n"
                           f"Action for Player: {current_role.upper()}"),
                "output": f"{cot}\nMove: {move_gdl}"
            })

            game.drop_piece(best_col, turn)
            turn = BLACK if turn == RED else RED

        if (game_idx + 1) % 50 == 0:
            print(f"  Partidas completadas: {game_idx + 1}/{num_games}  "
                  f"| Ejemplos acumulados: {len(dataset)}")

    return dataset

# --- Generar y guardar ---
print("Generando dataset Connect4")
new_data = generate_dataset(num_games=250, depth=4)

output_file = "Synthetic_Expert_Connect4_GDL.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for entry in new_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\nDataset guardado: {output_file}")
print(f"Total de ejemplos: {len(new_data)}")

Generando dataset Connect4
  Partidas completadas: 50/250  | Ejemplos acumulados: 1315
  Partidas completadas: 100/250  | Ejemplos acumulados: 2591
  Partidas completadas: 150/250  | Ejemplos acumulados: 3811
  Partidas completadas: 200/250  | Ejemplos acumulados: 5090
  Partidas completadas: 250/250  | Ejemplos acumulados: 6243

Dataset guardado: Synthetic_Expert_Connect4_GDL.jsonl
Total de ejemplos: 6243


**Unión de jugadas de "victoria" y "defensa" de CSV a Minimax**

In [ ]:
import json
import random

CATEGORY_KEYWORDS = {
    "victory":   "completes the line of four to achieve goal",
    "defensive": "blocks the threat",
}

def classify(entry):
    output = entry.get("output", "").lower()
    for cat, kw in CATEGORY_KEYWORDS.items():
        if kw in output:
            return cat
    return None

def load_jsonl(path):
    entries = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries

# Cargar ambos datasets
minimax = load_jsonl("Synthetic_Expert_Connect4_GDL.jsonl")
d1      = load_jsonl("D1_Expert_Connect4.jsonl")

# Extraer solo victory y defensive de D1
extras = [e for e in d1 if classify(e) in ("victory", "defensive")]

# Combinar y mezclar
merged = minimax + extras

# Guardar
with open("D3_Merged_Connect4.jsonl", "w", encoding="utf-8") as f:
    for entry in merged:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Reporte
print(f"Minimax original : {len(minimax):>5} entradas")
print(f"Victory + Def D1 : {len(extras):>5} entradas")
print(f"Total final      : {len(merged):>5} entradas")

Minimax original :  6243 entradas
Victory + Def D1 :  1725 entradas
Total final      :  7968 entradas


**Creación de dataset final**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   Connect4 TVN — Muestreador estratificado                           ║
# ║   Extrae N muestras del dataset generado                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────
# 1 — Subir el archivo fuente
# ─────────────────────────────────────────────────────────────────────
from google.colab import files

SOURCE_FILE = "dataset_final_filtrado.jsonl"
print("✓ Archivo recibido:")

# ─────────────────────────────────────────────────────────────────────
# CELDA 2 — Configuración
# ─────────────────────────────────────────────────────────────────────
TOTAL_SAMPLES = 3000
RANDOM_SEED   = 42
OUTPUT_FILE   = "Expert_Dataset_Connect4_3000.jsonl"

TARGET_FRACTIONS = {
    "victory":           0.40,
    "strategic_center":  0.20,
    "defensive":         0.30,
    "neutral":           0.10,
}

print("✓ Configuración lista")
for cat, frac in TARGET_FRACTIONS.items():
    n = round(TOTAL_SAMPLES * frac)
    print(f"  {cat:<22} {frac*100:.0f}%  →  {n} muestras")

# ─────────────────────────────────────────────────────────────────────
# 3 — Clasificación de entradas
# ─────────────────────────────────────────────────────────────────────
import json
import random
from collections import defaultdict

# Palabras clave
CATEGORY_KEYWORDS = {
    "victory":          ["winning opportunity", "completes the line of four"],
    "defensive":        ["threatening to complete", "blocks the threat"],
    "strategic_center": ["central zone", "belongs to the central axis"],
    "neutral":          ["analyzing gdl board predicates"]
}

def classify(entry: dict) -> str:
    # 1. Convertimos el output a minúsculas para que no falle por mayúsculas
    output = entry.get("output", "").lower()

    # 2. 'keywords' ahora es una lista, así que iteramos sobre ella
    for cat, keywords in CATEGORY_KEYWORDS.items():
        # 3. any() verifica si ALGUNA de las frases de la lista está en el texto
        if any(kw.lower() in output for kw in keywords):
            return cat

    return "neutral"  # fallback

# Cargar y clasificar
all_entries = []
with open(SOURCE_FILE, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            all_entries.append(json.loads(line))

buckets = defaultdict(list)
for entry in all_entries:
    buckets[classify(entry)].append(entry)

print(f"Total de entradas cargadas: {len(all_entries)}\n")
print(f"{'Categoría':<22} {'Entradas':>9}  {'%':>6}")
print("─" * 42)
for cat in TARGET_FRACTIONS:
    n   = len(buckets[cat])
    pct = n / len(all_entries) * 100
    print(f"  {cat:<20} {n:>9}  {pct:>5.1f}%")

# ─────────────────────────────────────────────────────────────────────
# 4 — Muestreo estratificado con verificación
# ─────────────────────────────────────────────────────────────────────
rng = random.Random(RANDOM_SEED)

# Calcular targets exactos (corregir redondeo)
targets = {cat: round(TOTAL_SAMPLES * frac) for cat, frac in TARGET_FRACTIONS.items()}
diff = TOTAL_SAMPLES - sum(targets.values())
targets["victory"] += diff   # Absorber diferencia de redondeo en el bucket más grande

print(f"Plan de muestreo para {TOTAL_SAMPLES} ejemplos:")
print(f"\n  {'Categoría':<22} {'Target':>7}  {'%':>5}  {'Disponibles':>12}  Estado")
print("  " + "─" * 65)

for cat, t in targets.items():
    avail  = len(buckets[cat])
    pct    = t / TOTAL_SAMPLES * 100
    status = "✓" if avail >= t else f"⚠  solo {avail} — se toman todos"
    print(f"  {cat:<22} {t:>7}  {pct:>4.0f}%  {avail:>12}  {status}")

# Muestrear
sampled = []
actual  = {}
for cat, t in targets.items():
    pool   = buckets[cat]
    n      = min(t, len(pool))
    chosen = rng.sample(pool, n)
    sampled.extend(chosen)
    actual[cat] = n

rng.shuffle(sampled)

# ─────────────────────────────────────────────────────────────────────
# 5 — Guardar y verificar
# ─────────────────────────────────────────────────────────────────────
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for entry in sampled:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✓ Dataset guardado: {OUTPUT_FILE}")
print(f"  Total de entradas: {len(sampled)}\n")
print("Composición final:")
print(f"  {'Categoría':<22} {'N':>5}  {'%':>6}")
print("  " + "─" * 38)
for cat, n in actual.items():
    pct = n / len(sampled) * 100
    print(f"  {cat:<22} {n:>5}  {pct:>5.1f}%")

# Verificación rápida: mostrar 1 ejemplo de cada categoría
print("\n── Muestra de 1 ejemplo por categoría ──")
shown = defaultdict(bool)
for entry in sampled:
    cat = classify(entry)
    if not shown[cat]:
        output_preview = entry["output"][:120].replace("\n", " ")
        print(f"\n[{cat}]\n  {output_preview}...")
        shown[cat] = True
    if all(shown[c] for c in TARGET_FRACTIONS):
        break

# ─────────────────────────────────────────────────────────────────────
# 6 — Guardar el archivo
# ─────────────────────────────────────────────────────────────────────
import os
import shutil
from google.colab import drive
# 1. Conectar Google Drive
drive.mount('/content/drive')

# 2. Definir rutas
archivo_en_colab = OUTPUT_FILE
carpeta_en_drive = "/content/drive/MyDrive/TESIS/Qwen_Connect4_NotConnect4/TVN/"

# 3. Crear la carpeta en Drive si no existe
os.makedirs(carpeta_en_drive, exist_ok=True)

# 4. Copiar el archivo
ruta_destino = os.path.join(carpeta_en_drive, os.path.basename(archivo_en_colab))
shutil.copy(archivo_en_colab, ruta_destino)

print(f"¡Archivo copiado exitosamente a: {ruta_destino}!")

✓ Archivo recibido:
✓ Configuración lista
  victory                30%  →  90 muestras
  strategic_center       20%  →  60 muestras
  defensive              40%  →  120 muestras
  neutral                10%  →  30 muestras
Total de entradas cargadas: 4241

Categoría               Entradas       %
──────────────────────────────────────────
  victory                     86    2.0%
  strategic_center          1880   44.3%
  defensive                  221    5.2%
  neutral                   2054   48.4%
Plan de muestreo para 300 ejemplos:

  Categoría               Target      %   Disponibles  Estado
  ─────────────────────────────────────────────────────────────────
  victory                     90    30%            86  ⚠  solo 86 — se toman todos
  strategic_center            60    20%          1880  ✓
  defensive                  120    40%           221  ✓
  neutral                     30    10%          2054  ✓
✓ Dataset guardado: Expert_Dataset_Connect4_Test_300.jsonl
  Total de entr